# HomeHand — fine-tune SmolVLA on Kaggle

Settings (right panel): **Accelerator: GPU T4 x2** (not P100: recent PyTorch builds no longer support it),
**Internet: on** (needs a phone-verified account), **Persistence: Files only**.
Add the uploaded dataset `homehand-vla-v1` as input (it contains the folder `vla_v1`).

Run with **Save Version → Save & Run All (Commit)**: the run continues in the background for up to 12 h and
everything in `/kaggle/working` (checkpoints) becomes the version's output.

In [ ]:
!pip install -q "lerobot[dataset,smolvla,training]==0.6.1"
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0))
# SmolVLA loads its VLM in bfloat16, which the T4 has no hardware for (emulated: > 7 s/step, and AMP
# cannot unscale bf16 gradients): load it in float32 and let AMP run the matmuls in fp16 on the tensor cores
import lerobot, os
p = os.path.join(os.path.dirname(lerobot.__file__), 'policies', 'smolvla', 'smolvlm_with_expert.py')
s = open(p).read(); assert 'torch_dtype="bfloat16"' in s
open(p, 'w').write(s.replace('torch_dtype="bfloat16"', 'torch_dtype="float32"')); print('patched', p)

In [ ]:
import glob, os, shutil
src = glob.glob('/kaggle/input/**/vla_v1/meta/info.json', recursive=True)
assert src, 'dataset not found: add the homehand-vla-v1 dataset as input'
root = os.path.dirname(os.path.dirname(src[0]))
DATA = '/kaggle/working/vla_v1'
if not os.path.exists(DATA):
    shutil.copytree(root, DATA)          # LeRobot may write caches next to the data; the input is read-only
print(DATA, os.listdir(DATA))

In [ ]:
OUT = '/kaggle/working/outputs/smolvla_homehand'
STEPS, BATCH = 3000, 16                   # fp16 on the T4; measured fp32 was > 7 s/step
last = f'{OUT}/checkpoints/last/pretrained_model/train_config.json'
if os.path.exists(last):                  # resuming a previous run (copy its outputs into /kaggle/working first)
    cmd = f'lerobot-train --config_path={last} --resume=true'
else:
    cmd = (f'lerobot-train --policy.path=lerobot/smolvla_base --dataset.repo_id=homehand/vla_v1 '
           f'--dataset.root={DATA} --batch_size={BATCH} --steps={STEPS} --save_freq=1000 --log_freq=25 '
           f'--output_dir={OUT} --job_name=smolvla_homehand --policy.device=cuda '
           f'--policy.push_to_hub=false --wandb.enable=false --num_workers=4 --policy.use_amp=true')
print(cmd)
!{cmd}

In [ ]:
# keep only the final weights for download (the full checkpoints include optimizer states)
import shutil
final = f'{OUT}/checkpoints/last/pretrained_model'
shutil.make_archive('/kaggle/working/smolvla_homehand', 'zip', final)
print(os.listdir(final))